# Nemotron diagnosis

Nemotron interprets evidence through read-only tools and proposes one bounded experiment. Python alone records measurements, admits controls, and decides PASS/FAIL/INCONCLUSIVE.

Nemotron may interpret evidence, generate hypotheses, identify missing evidence, and propose one experiment. It may not change measurements, declare PASS/FAIL, invent GPU metrics, or bypass deterministic policy.

In [ ]:
import os
from pathlib import Path
from inferdoc.config import InferDocSettings
from inferdoc.evidence.models import EvidenceBundle
from inferdoc.doctor.agent import DoctorAgent
from inferdoc.doctor.models import DiagnosisReport
from inferdoc.doctor.tools import deterministic_tools, tool_definitions
from inferdoc.experiments.capabilities import DEMO_TOKEN_FACTORY_CAPABILITIES
from inferdoc.experiments.policy import validate_experiment
from inferdoc.nebius.client import NebiusClient

settings = InferDocSettings.from_env()
live_enabled = os.getenv("INFERDOC_RUN_LIVE_TESTS") == "1" and bool(settings.api_key)
demo_dir = Path("examples/demo_artifacts")
baseline = EvidenceBundle.model_validate_json((demo_dir / "baseline.json").read_text(encoding="utf-8"))
diagnosis = DiagnosisReport.model_validate_json((demo_dir / "diagnosis.json").read_text(encoding="utf-8"))
print({"baseline_run_id": baseline.run_id, "live_enabled": live_enabled})

## Read-only evidence tools

The same tool definitions and deterministic responses are used by `DoctorAgent`. The sanitized diagnosis omits captured tool call IDs; a new live diagnosis populates its `audit_log`.

In [ ]:
tools = deterministic_tools(baseline, DEMO_TOKEN_FACTORY_CAPABILITIES)
print("tools:", [item["function"]["name"] for item in tool_definitions()])
print("get_run_summary:", tools["get_run_summary"]()["aggregate"])
print("get_request_metrics count:", len(tools["get_request_metrics"]()["requests"]))
print("get_backend_capabilities:", tools["get_backend_capabilities"]())

## Captured DiagnosisReport and optional new Nemotron call

The saved report came from a real Token Factory run. Set `RUN_LIVE_DIAGNOSIS=True` as well as the live gate to ask Nemotron to analyze the saved evidence again.

In [ ]:
RUN_LIVE_DIAGNOSIS = False
if RUN_LIVE_DIAGNOSIS and live_enabled:
    client = NebiusClient(settings=settings)
    try:
        diagnosis = await DoctorAgent(
            client, capabilities=DEMO_TOKEN_FACTORY_CAPABILITIES
        ).adiagnose(baseline)
    finally:
        await client.aclose()
else:
    print("Using the captured sanitized DiagnosisReport; no Nemotron call.")
print({"measured_facts": diagnosis.measured_facts,
       "hypotheses": diagnosis.hypotheses,
       "missing_evidence": diagnosis.missing_evidence,
       "recommendation": diagnosis.recommendation_summary,
       "experiment": diagnosis.experiment.model_dump(mode="json") if diagnosis.experiment else None,
       "audit_log": diagnosis.audit_log})

## Deterministic admission

Nemotron proposes; `validate_experiment` decides whether the backend controls and requested measurements are allowed.

In [ ]:
if diagnosis.experiment is not None:
    admission = validate_experiment(
        diagnosis.experiment, DEMO_TOKEN_FACTORY_CAPABILITIES,
        baseline=baseline, max_changed_variables=1,
    )
    print(admission.model_dump(mode="json"))
else:
    print("No executable experiment was proposed; this is a valid outcome.")